In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_49.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_90.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_77.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_66.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_54.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_42.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_81.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_72.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final

In [2]:
import os
import numpy as np
import pandas as pd
import librosa

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
DATA_PATH = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

TRAIN_AUDIO_PATH = f"{DATA_PATH}/train"
TEST_AUDIO_PATH = f"{DATA_PATH}/test"

train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nTrain columns:", list(train_df.columns))
print("Test columns :", list(test_df.columns))

Train shape: (769, 2)
Test shape : (216, 2)

Train columns: ['filename', 'label']
Test columns : ['filename', 'label']


In [4]:
def extract_audio_features(filepath):
    audio, sr = librosa.load(filepath, sr=None)

    features = []

    # MFCC features
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=13,
        n_fft=2048,
        hop_length=512
    )

    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # RMS energy
    rms = librosa.feature.rms(
        y=audio,
        frame_length=2048,
        hop_length=512
    )

    features.append(np.mean(rms))
    features.append(np.std(rms))

    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(
        audio,
        frame_length=2048,
        hop_length=512
    )

    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    # Spectral features
    spectral_centroid = librosa.feature.spectral_centroid(
        y=audio,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    spectral_bandwidth = librosa.feature.spectral_bandwidth(
        y=audio,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    spectral_rolloff = librosa.feature.spectral_rolloff(
        y=audio,
        sr=sr,
        n_fft=2048,
        hop_length=512
    )

    features.extend([
        np.mean(spectral_centroid),
        np.std(spectral_centroid),
        np.mean(spectral_bandwidth),
        np.std(spectral_bandwidth),
        np.mean(spectral_rolloff),
        np.std(spectral_rolloff)
    ])

    # Audio duration
    features.append(len(audio) / sr)

    return np.array(features)


print("Feature extraction function ready.")
print("Number of features:", len(extract_audio_features(
    os.path.join(TRAIN_AUDIO_PATH, train_df.iloc[0]["filename"])
)))

Feature extraction function ready.
Number of features: 37


In [5]:
train_features = []

for filename in tqdm(train_df["filename"], desc="Extracting training features"):
    filepath = os.path.join(TRAIN_AUDIO_PATH, filename)
    features = extract_audio_features(filepath)
    train_features.append(features)

train_features = np.array(train_features)

print("\nTraining feature extraction complete.")
print("Feature matrix shape:", train_features.shape)

Extracting training features:   0%|          | 0/769 [00:00<?, ?it/s]


Training feature extraction complete.
Feature matrix shape: (769, 37)


In [6]:
feature_columns = (
    [f"mfcc_mean_{i+1}" for i in range(13)] +
    [f"mfcc_std_{i+1}" for i in range(13)] +
    [
        "rms_mean",
        "rms_std",
        "zcr_mean",
        "zcr_std",
        "spectral_centroid_mean",
        "spectral_centroid_std",
        "spectral_bandwidth_mean",
        "spectral_bandwidth_std",
        "spectral_rolloff_mean",
        "spectral_rolloff_std",
        "duration"
    ]
)

X = pd.DataFrame(train_features, columns=feature_columns)
y = train_df["label"].copy()

print("Feature DataFrame shape:", X.shape)
print("Target shape:", y.shape)
print("\nNumber of features:", len(feature_columns))

Feature DataFrame shape: (769, 37)
Target shape: (769,)

Number of features: 37


In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples  :", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Features          :", X_train.shape[1])

Training samples  : 615
Validation samples: 154
Features          : 37


In [8]:
model = ExtraTreesRegressor(
    n_estimators=500,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Extra Trees model trained successfully.")

Extra Trees model trained successfully.


In [9]:
val_predictions = model.predict(X_val)

val_rmse = np.sqrt(
    mean_squared_error(y_val, val_predictions)
)

val_pearson = pearsonr(y_val, val_predictions)[0]

print(f"Validation RMSE   : {val_rmse:.4f}")
print(f"Validation Pearson: {val_pearson:.4f}")

Validation RMSE   : 0.6823
Validation Pearson: 0.8656


In [10]:
final_model = ExtraTreesRegressor(
    n_estimators=500,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

print("Final model trained successfully.")
print("Training samples:", len(X))
print("Features:", X.shape[1])

Final model trained successfully.
Training samples: 769
Features: 37


In [11]:
train_predictions = final_model.predict(X)

train_rmse = np.sqrt(
    mean_squared_error(y, train_predictions)
)

print(f"Training RMSE: {train_rmse:.4f}")

Training RMSE: 0.0000


In [12]:
test_features = []

for filename in tqdm(test_df["filename"], desc="Extracting test features"):
    filepath = os.path.join(TEST_AUDIO_PATH, filename)
    features = extract_audio_features(filepath)
    test_features.append(features)

test_features = np.array(test_features)

print("\nTest feature extraction complete.")
print("Test feature matrix shape:", test_features.shape)

Extracting test features:   0%|          | 0/216 [00:00<?, ?it/s]


Test feature extraction complete.
Test feature matrix shape: (216, 37)


In [13]:
test_predictions = final_model.predict(test_features)

print("Predictions generated successfully.")
print("Number of predictions:", len(test_predictions))

print("\nPrediction statistics:")
print("Min :", test_predictions.min())
print("Max :", test_predictions.max())
print("Mean:", test_predictions.mean())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(


Predictions generated successfully.
Number of predictions: 216

Prediction statistics:
Min : 2.484
Max : 4.212
Mean: 3.2350277777777783


In [14]:
X_test = pd.DataFrame(
    test_features,
    columns=feature_columns
)

test_predictions = final_model.predict(X_test)

print("Test predictions generated successfully.")
print("Prediction count:", len(test_predictions))
print("Prediction range:", test_predictions.min(), "to", test_predictions.max())

Test predictions generated successfully.
Prediction count: 216
Prediction range: 2.484 to 4.212


In [15]:
submission_df = pd.DataFrame({
    "filename": test_df["filename"],
    "label": np.clip(test_predictions, 0, 5)
})

submission_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_path, index=False)

print("Submission created successfully.")
print("Shape:", submission_df.shape)
print("\nFirst 10 predictions:")
print(submission_df.head(10))
print("\nSaved to:", submission_path)

Submission created successfully.
Shape: (216, 2)

First 10 predictions:
        filename  label
0  audio_128.wav  3.165
1  audio_156.wav  4.042
2   audio_19.wav  3.322
3  audio_108.wav  2.514
4  audio_206.wav  3.325
5  audio_161.wav  2.967
6  audio_215.wav  3.003
7  audio_213.wav  3.007
8   audio_11.wav  3.347
9  audio_155.wav  3.289

Saved to: /kaggle/working/submission.csv


In [16]:
check = pd.read_csv("/kaggle/working/submission.csv")

print("===== FINAL SUBMISSION CHECK =====")
print("Rows:", len(check))
print("Columns:", list(check.columns))

print("\nMissing values:")
print(check.isnull().sum())

print("\nDuplicate filenames:", check["filename"].duplicated().sum())

print("\nPrediction range:")
print("Min:", check["label"].min())
print("Max:", check["label"].max())

print("\nFilename order matches test.csv:",
      check["filename"].tolist() == test_df["filename"].tolist())

print("\n===== FINAL MODEL METRICS =====")
print(f"Validation RMSE   : {val_rmse:.4f}")
print(f"Validation Pearson: {val_pearson:.4f}")
print(f"Training RMSE     : {train_rmse:.4f}")

print("\n===== STATUS =====")
if (
    len(check) == len(test_df)
    and check["filename"].duplicated().sum() == 0
    and check.isnull().sum().sum() == 0
    and check["filename"].tolist() == test_df["filename"].tolist()
    and check["label"].between(0, 5).all()
):
    print("✓ SUBMISSION IS VALID")
else:
    print("✗ CHECK SUBMISSION")

===== FINAL SUBMISSION CHECK =====
Rows: 216
Columns: ['filename', 'label']

Missing values:
filename    0
label       0
dtype: int64

Duplicate filenames: 0

Prediction range:
Min: 2.484
Max: 4.212

Filename order matches test.csv: True

===== FINAL MODEL METRICS =====
Validation RMSE   : 0.6823
Validation Pearson: 0.8656
Training RMSE     : 0.0000

===== STATUS =====
✓ SUBMISSION IS VALID
